# Timing jitter of solitons and adjoint approach to Langevin equations

The example analyzed here is timing jitter of solitons induced by vacuum fluctuations, amplifier noise (Gordon-Haus effect), and Raman scattering. An analytical treatment of this problem is treated in Corney and Drummond (https://opg.optica.org/josab/fulltext.cfm?uri=JOSAB-18-2-153).

In [ ]:
using Pkg

const ROOT = abspath(joinpath(@__DIR__, ".."))
Pkg.activate(ROOT)

using LinearAlgebra
using Printf
using Random
using Statistics
using FFTW
using PyPlot
using DelimitedFiles

isdefined(Main, :PulsePropagation) ||
    include(joinpath(ROOT, "src", "PulsePropagation.jl"))
using .PulsePropagation

const c_light_m_ps = 2.99792458e-4
const c_light_m_s = 2.99792458e8
const hbar = 1.054571817e-34
const kb = 1.380649e-23

In [ ]:
## Grid and simulation properties
Nt::Int = 2^15;
time_window_ps::Float64 = 200.0;
lambda0_m::Float64 = 1550e-9;
f0 = c_light_m_s / lambda0_m * 1e-12; ## THz

grid = TimeGrid(Nt, time_window_ps)

t = time_axis(grid)                         # ps, centered
f = frequency_axis(grid; shifted=true)      # THz relative frequency grid
λ = wavelength_axis(grid, f0; shifted=true) # nm
w = photon_weights(grid, f0; shifted=false) # photon-bin weights

## Fiber properties
L_fiber::Float64 = 500;

beta2_si::Float64 = -2.04e-26;

beta2_ps2_m::Float64 = -2.04e-26 * 1e24;
betas = reshape([0.0, 0.0, beta2_ps2_m], :, 1);

n2::Float64 = 2.3e-20;
mfd_um::Float64 = 9.2;
Aeff_m2::Float64 = π * (mfd_um * 1e-6 / 2)^2;
sr = reshape([1 / Aeff_m2], 1, 1, 1, 1);
gamma = n2*2*π/Aeff_m2/lambda0_m;


f_raman::Float64 = 0.18;
tau1_raman_fs::Float64 = 12.2;
tau2_raman_fs::Float64 = 32.0;
temperature::Float64 = 300.0;

system = PassiveFiber(; length = L_fiber, lambda0 = lambda0_m, betas = betas, f0 = f0, material=AgarwalRaman(; fraction=f_raman, tau1_fs=tau1_raman_fs, tau2_fs=tau2_raman_fs, temperature_K=temperature), sr = sr, mfd = mfd_um, n2 = n2, raman_fraction = f_raman, source_path = nothing)
dofs   = SingleModeField()
terms = PropagationTerms(Dispersion(), Kerr(), Raman());

## Pulse properties
pulse_fwhm_ps::Float64 = 4; #ps
tau_sech = 4 / 1.76; 
L_disp = tau_sech^2 / abs(beta2_ps2_m);


peak_power = 1 * abs(beta2_si) / (gamma) / (tau_sech * 1e-12)^2;

fields = sech_pulse(grid; peak_power = peak_power, fwhm=pulse_fwhm_ps,
                    dofs=dofs,
                    transform=("ifft", 0.0),
                    time_offset=0.0);

In [ ]:
figure(figsize=(3,3))
plot(t, abs2.(fields[:,1,1]))
xlim(-10,10)
display(gcf())

In [ ]:
sum(abs2.(fields[:,1,1])) * diff(t)[1]

In [ ]:
N_z = 1501;
zsave = collect(range(0.0, L_fiber; length=N_z))

solver = RK4IPSolver(;
    stepping = AdaptiveStep(
        initial_dz = 1,
        max_dz = 100,
        rtol = 1e-6,
        atol = 1e-6,
    ),
    saveat = SaveAt(zsave),
)

In [ ]:
problem = PulsePropagationProblem(; grid, system, dofs, terms, solver, fields)
sol = solve(problem)

In [ ]:
spec_out = spectrum(sol);
spec_z = z_spectrum(sol; shifted=true, sum_modes=false);
n0 = sum(fftshift(w) .* spec_z[:,1,1]);

figure(figsize=(3,3))
plot(λ, spec_z[:,1,1])
plot(λ, spec_z[:,1,end],"--")
xlim(1548, 1552)
display(gcf())

In [ ]:
figure(figsize=(3,3))
contourf(λ, sol.output.z, 10log10.(spec_z[:,1,:]'), levels=[-75:2.5:0;])
colorbar()
xlabel("Wavelength (nm)")
ylabel("z (m)")
xlim(1548,1552)
clim(-40,0)
display(gcf())

In [ ]:
z_list = [0.1:0.25:8.1;]*L_disp;

gain = 0.001;
tau_sech_s = tau_sech * 1e-12;


tau_sech = pulse_fwhm_ps / (2asech(1 / sqrt(2)))
F0 = 2 * kb * temperature / hbar *
     (tau2_raman_fs * 1e-15) /
     (1 + tau2_raman_fs^2 / tau1_raman_fs^2) *
     f_raman

t_vac = zero(z_list);
t_gh = zero(z_list);
t_raman = zero(z_list);

t_vac_drummond = zero(z_list);
t_gh_drummond = zero(z_list);
t_raman_drummond = zero(z_list);

Cω = PulsePropagation.raman_langevin_spectrum(
    Nt, diff(t)[1], f0, gamma;
    raman_fraction=0.18,
    temperature_K=300.0,
);

for ii = 1:length(z_list)
    L_fiber = z_list[ii];
    t_vac_drummond[ii] = (tau_sech_s)^2 * ((π^2)/12 + 1/3 *(L_fiber/L_disp)^2) / n0

    t_gh_d_lin = (tau_sech_s)^2 * ((gain/2) * 2 * π^2 / 12 / n0) * (1/L_disp)^1 * L_disp .* (L_fiber).^1;
    t_gh_d_cubic = (tau_sech_s)^2 * ((gain/2) * 2 / 9 / n0) * (1/L_disp)^3 * L_disp .* (L_fiber).^3;
    t_gh_drummond[ii] = t_gh_d_lin + t_gh_d_cubic;

    
    t_raman_drummond[ii] = (16/45)*(tau_sech_s)^2 * (F0) .* (L_fiber/L_disp).^3 / (n0)
end

figure(figsize=(3,3))
semilogy(z_list/L_disp,t_vac_drummond)
semilogy(z_list/L_disp,t_gh_drummond)
semilogy(z_list/L_disp,t_raman_drummond)
display(gcf())

In [ ]:
for ii = 1:length(z_list)
    L_fiber = z_list[ii];
    
    # system instantiation
    system = PassiveFiber(; length = L_fiber, lambda0 = lambda0_m, betas = betas, f0 = f0, material=AgarwalRaman(; fraction=f_raman, tau1_fs=tau1_raman_fs, tau2_fs=tau2_raman_fs, temperature_K=temperature), sr = sr, mfd = mfd_um, n2 = n2, raman_fraction = f_raman, source_path = nothing)
    zsave = collect(range(0.0, L_fiber; length=N_z))

    # solver
    solver = RK4IPSolver(;
    stepping = AdaptiveStep(
        initial_dz = 0.1,
        max_dz = 10,
        rtol = 1e-10,
        atol = 1e-10,
    ),
    saveat = SaveAt(zsave),
    )
    
    # forward solve
    problem = PulsePropagationProblem(; grid, system, dofs, terms, solver, fields)
    sol = solve(problem)

    # observable
    
    obs = TemporalMoment(;
    order = 1,
    center = 0.0,
    normalized = true,
    modes = 1,
    )

    grad = gradient(
    problem,
    obs;
    trajectory=sol,
    method=PulsePropagation.Adjoint(),
    wrt=InitialField(),
    normalization=PhotonNormalized(),
    raman=:agarwal,
    raman_fraction=0.18,
    dz_adj=10.0,
    reltol=1e-8,
    abstol=1e-8,
    )
    adj_photon = photon_normalized_adjoint_trajectory(grad, problem);
    
    # Vacuum fluctuation contribution
    t_vac[ii] = sum(abs2.(grad.initial_field_gradient)) * 1e-24;    

    # Gordon-Haus contribution
    dz = diff(grad.adjoint_trajectory.z)[1];
    t_gh[ii] = gain*(sum(abs2.(adj_photon.lambdaw))) * 1e-24 * dz

    # Raman contribution
    Azt = sol.output.fields;
    λzt = conj(fft(grad.adjoint_trajectory.lambdaw,1));
    Bzt = -2*imag(Azt .* λzt);
    Bzω = ifft(Bzt,1);
    t_raman[ii] = gamma^2 * sum(abs2.(Bzω) .* Cω) * (diff(f)[1] * 1e12) * 1e-24 * (L_fiber / N_z);
    
end

In [ ]:
figure(figsize=(5,2.5))

vac_qsa, = semilogy(z_list / L_disp,t_vac,"o")
gh_qsa, = semilogy(z_list / L_disp,t_gh,"x")
raman_qsa, = semilogy(z_list / L_disp,t_raman,"^")

vac_spt, = semilogy(z_list / L_disp,t_vac_drummond)
gh_spt, = semilogy(z_list / L_disp,t_gh_drummond)
raman_spt, = semilogy(z_list / L_disp,t_raman_drummond)

xlabel("L/Ld")
ylabel("Jitter (s²)")
legend([vac_qsa, vac_spt, gh_qsa, gh_spt, raman_qsa, raman_spt],
       ["Vacuum (QSA)", "Vacuum (SPT)", "Gordon-Haus (QSA)", "Gordon-Haus (SPT)", "Raman (QSA)", "Raman (SPT)"],
       loc="lower right",  fontsize=8)
tight_layout()

display(gcf())

In [ ]:
figure(figsize=(3,3))
plot(fftshift(Cω))
display(gcf())

In [ ]:
figure(figsize=(3,3))
plot(z_list / L_disp,t_vac ./ t_vac_drummond)
plot(z_list / L_disp,t_gh ./ t_gh_drummond)
plot(z_list / L_disp,t_raman ./ t_raman_drummond)
display(gcf())

<!-- opt_filt_trunc = readdlm(joinpath(@__DIR__, "optimal_filter.csv"));
opt_filt = zeros(p.Nt);
opt_filt[1196:2371] = reverse(opt_filt_trunc); 

λ_grid_adj = λ[1196:2371]; # hard coded numbers based on example - has to do with computing Jacobian (∂n_i/∂u_j) not over full wavelength band of simulation, which is very expensive and most wavelengths don't participate in dynamics
ww_index_grid = unique([argmin(abs.(fftshift(λ) .- ll)) for ll in λ_grid_adj]);
ww_index_grid = reverse(ww_index_grid); -->